In [ ]:
import numpy as np
import onnxruntime as ort
import matplotlib.pyplot as plt
from pathlib import Path
import time
from copy import deepcopy

# For reproducibility
np.random.seed(42)

In [ ]:
class ConnectFour:
    """
    Simple board representation.
    board: 2D numpy array (9 rows, 10 cols), 0 = empty, 1 = player 0, 2 = player 1.
    """
    def __init__(self, rows=9, cols=10, inarow=5):
        self.rows = rows
        self.cols = cols
        self.inarow = inarow
        self.board = np.zeros((rows, cols), dtype=np.int8)
        self.current_player = 0
        self.done = False
        self.winner = None

    def clone(self):
        new = ConnectFour(self.rows, self.cols, self.inarow)
        new.board = self.board.copy()
        new.current_player = self.current_player
        new.done = self.done
        new.winner = self.winner
        return new

    def legal_moves(self):
        """Return list of columns where top cell is empty."""
        if self.done:
            return []
        return [c for c in range(self.cols) if self.board[0, c] == 0]

    def apply_move(self, col):
        """Place piece for current player in column col. Returns True if move is legal."""
        if col not in self.legal_moves():
            return False
        for r in range(self.rows-1, -1, -1):
            if self.board[r, col] == 0:
                self.board[r, col] = self.current_player + 1  # 1 or 2
                break
        self._check_termination(r, col)
        self.current_player = 1 - self.current_player
        return True

    def _check_termination(self, row, col):
        """Check if the last move caused a win."""
        player = self.board[row, col]
        directions = [(0,1), (1,0), (1,1), (1,-1)]
        for dr, dc in directions:
            count = 1
            # positive direction
            r, c = row + dr, col + dc
            while 0 <= r < self.rows and 0 <= c < self.cols and self.board[r,c] == player:
                count += 1
                r += dr
                c += dc
            # negative direction
            r, c = row - dr, col - dc
            while 0 <= r < self.rows and 0 <= c < self.cols and self.board[r,c] == player:
                count += 1
                r -= dr
                c -= dc
            if count >= self.inarow:
                self.done = True
                self.winner = player - 1  # 0 or 1
                return
        if len(self.legal_moves()) == 0:
            self.done = True
            self.winner = None  # draw

    def display(self):
        symbols = {0: '.', 1: 'X', 2: 'O'}
        for r in range(self.rows):
            print(' '.join(symbols[self.board[r,c]] for c in range(self.cols)))
        print()

In [ ]:
class ValueNetPredictor:
    """
    Loads the ONNX model and provides a method `predict(board, player)` returning
    a value in [-1,1] from the given player's perspective.
    Input board must be a (2,9,10) tensor with channel 0 = current player's pieces.
    """
    def __init__(self, onnx_path="../ml/models/value_net.onnx"):
        self.session = ort.InferenceSession(str(onnx_path))
        self.input_name = self.session.get_inputs()[0].name

    def predict(self, board_state, player):
        """
        board_state: (9,10) numpy array of 0 (empty), 1 (player0), 2 (player1).
        player: whose perspective we want.
        Returns scalar float in [-1,1].
        """
        # Convert to binary tensor (2,9,10)
        tensor = np.zeros((1, 2, 9, 10), dtype=np.float32)
        # channel 0 = current player (the one we want perspective)
        # If player wants perspective of player 0:
        #   positions with 1 -> channel0, positions with 2 -> channel1
        # If player wants perspective of player 1:
        #   positions with 2 -> channel0, positions with 1 -> channel1
        for r in range(9):
            for c in range(10):
                cell = board_state[r, c]
                if cell == player + 1:   # player's own piece (player+1 maps 0->1, 1->2)
                    tensor[0, 0, r, c] = 1.0
                elif cell != 0:          # opponent's piece
                    tensor[0, 1, r, c] = 1.0
        out = self.session.run(None, {self.input_name: tensor})[0]
        return float(out[0])

In [ ]:
class MCTSNode:
    def __init__(self, state: ConnectFour, parent=None, action=None):
        self.state = state
        self.parent = parent
        self.action = action
        self.children = {}
        self.visit_count = 0
        self.total_value = 0.0
        self.untried_actions = state.legal_moves()

    def is_fully_expanded(self):
        return len(self.untried_actions) == 0

    def best_child(self, c_param=1.41):
        best_score = -float('inf')
        best_child = None
        for child in self.children.values():
            if child.visit_count == 0:
                ucb = float('inf')
            else:
                mean = child.total_value / child.visit_count
                ucb = mean + c_param * np.sqrt(np.log(self.visit_count) / child.visit_count)
            if ucb > best_score:
                best_score = ucb
                best_child = child
        return best_child

    def backpropagate(self, value):
        node = self
        while node is not None:
            node.visit_count += 1
            node.total_value += value
            value = -value  # flip for parent
            node = node.parent

In [ ]:
def evaluate_terminal(state, player_to_move):
    """Return value in [-1,1] for player_to_move in terminal state."""
    if state.winner is None:
        return 0.0
    # state.winner is the player who won (0 or 1). player_to_move is the next player (who would move).
    # If winner == player_to_move -> impossible because the winner already made the last move, and game ended before opponent's turn.
    # So winner != player_to_move. Thus if winner == player_to_move, it would mean the player to move is the winner (can't happen).
    # So we return 1 if player_to_move is the winner? No. The player who didn't win loses.
    return 1.0 if state.winner == player_to_move else -1.0

def mcts_search(root_state, value_net, num_simulations=800, c_param=1.41):
    root = MCTSNode(root_state)
    for sim in range(num_simulations):
        node = root
        # Traverse until a non-fully-expanded node or terminal
        while not node.state.done and node.is_fully_expanded():
            node = node.best_child(c_param)
        if node.state.done:
            # Terminal node – evaluate from the perspective of the player who is about to move (node's perspective)
            val = evaluate_terminal(node.state, node.state.current_player)
            node.backpropagate(val)
        else:
            # Expand
            col = np.random.choice(node.untried_actions)
            node.untried_actions.remove(col)
            # Create child state
            child_state = node.state.clone()
            child_state.apply_move(col)
            child = MCTSNode(child_state, parent=node, action=col)
            node.children[col] = child
            # Evaluate leaf
            if child.state.done:
                val = evaluate_terminal(child.state, child.state.current_player)
            else:
                val = value_net.predict(child.state.board, child.state.current_player)
            # Store leaf value
            child.visit_count = 1
            child.total_value = val
            # Backprop from child (value is from child's perspective)
            child.backpropagate(val)
    # Compute action probabilities
    visits = np.array([child.visit_count for col, child in root.children.items()])
    total = visits.sum()
    if total > 0:
        probs = visits / total
    else:
        probs = np.ones(root.state.cols) / root.state.cols  # fallback
    return probs, root

In [ ]:
class NNMCTSAgent:
    def __init__(self, value_net, num_simulations=800, c_param=1.41, temperature=0.1):
        self.value_net = value_net
        self.num_simulations = num_simulations
        self.c_param = c_param
        self.temperature = temperature

    def act(self, state: ConnectFour):
        probs, _ = mcts_search(state, self.value_net,
                               num_simulations=self.num_simulations,
                               c_param=self.c_param)
        # Apply temperature
        if self.temperature == 0:
            col = np.argmax(probs)
        else:
            probs = np.power(probs, 1.0 / self.temperature)
            probs /= probs.sum()
            col = np.random.choice(len(probs), p=probs)
        return col

In [ ]:
def play_game(agent1, agent2=None, verbose=True):
    """
    agent1 plays as player 0, agent2 as player 1.
    If agent2 is None, use random agent.
    """
    game = ConnectFour()
    if agent2 is None:
        agent2 = RandomAgent()
    if verbose:
        print("Starting game. X = NN-MCTS (player 0), O = Random (player 1)")
    move_count = 0
    while not game.done and move_count < 90:
        if verbose:
            print(f"\nMove {move_count+1}, Player {game.current_player} turn")
            game.display()
        if game.current_player == 0:
            col = agent1.act(game)
        else:
            col = agent2.act(game)
        if col not in game.legal_moves():
            if verbose:
                print(f"Illegal move {col} by player {game.current_player}, choosing random legal move")
            col = np.random.choice(game.legal_moves())
        game.apply_move(col)
        if verbose:
            print(f"Player {game.current_player} (just moved) chose column {col}")
        move_count += 1
    if verbose:
        print("\nFinal board:")
        game.display()
        if game.winner is None:
            print("Draw!")
        else:
            print(f"Winner: Player {game.winner} ({'X' if game.winner==0 else 'O'})")
    return game.winner

class RandomAgent:
    def act(self, state):
        return np.random.choice(state.legal_moves())

In [ ]:
# Load the value network
value_net = ValueNetPredictor("../ml/models/value_net.onnx")

# Create NN-MCTS agent (adjust num_simulations based on your patience)
nn_agent = NNMCTSAgent(value_net, num_simulations=200, c_param=1.41, temperature=0.1)

# Play one game (NN-MCTS vs random)
winner = play_game(nn_agent, verbose=True)